[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SivanSchwartz/representation-learning-course/blob/master/tutoring/session01/solutions/02_bridge_to_pytorch.ipynb)

# 02 · The bridge: `nn.Module` is the OOP you just wrote

Fell behind on 01? Run `git pull --no-edit` and open `tutoring/session01/solutions/01_oop.ipynb`.

In [1]:
# Setup: on Colab, install Lightning. Then load today's data.
import io
import os
import subprocess
import sys
import urllib.request

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytorch-lightning"], check=True)

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DATA_URL = "https://raw.githubusercontent.com/SivanSchwartz/representation-learning-course/master/tutoring/data/mnist_small.npz"


def load_mnist():
    """The local copy if there is one (your laptop), otherwise download it (Colab)."""
    for path in ["../data/mnist_small.npz", "../../data/mnist_small.npz"]:
        if os.path.exists(path):
            return dict(np.load(path))
    return dict(np.load(io.BytesIO(urllib.request.urlopen(DATA_URL).read())))


data = load_mnist()
torch.manual_seed(0)

## 1. lec1's model, for real this time

In [2]:
# ✍️ lec1's model. Compare it with TinyModule, Scale and Pipeline from notebook 01.
class NN(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 50)
        self.fc2 = nn.Linear(50, num_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)


model = NN(784, 10)
model

NN(
  (fc1): Linear(in_features=784, out_features=50, bias=True)
  (fc2): Linear(in_features=50, out_features=10, bias=True)
)

Printing the model shows the **tree** of modules stored on `self` (composition, section 5 of notebook 01). `nn.Module` also finds every weight in that tree for you:

In [3]:
# ✍️ parameters() walks the whole tree
for name, p in model.named_parameters():
    print(f"{name:12} {tuple(p.shape)}")
print("total:", sum(p.numel() for p in model.parameters()))

fc1.weight   (50, 784)
fc1.bias     (50,)
fc2.weight   (10, 50)
fc2.bias     (10,)
total: 39760


In [4]:
# 🤔 Predict: two instances of the same class. Same weights or different? (True or False?)
model_a = NN(784, 10)
model_b = NN(784, 10)
torch.equal(model_a.fc1.weight, model_b.fc1.weight)

False

In [5]:
# 🤔 Predict: model(x) and model.forward(x). Same result?
x = torch.rand(3, 784)
print(model(x).shape)
print(torch.equal(model(x), model.forward(x)))

torch.Size([3, 10])
True


Same result. `model(x)` goes through `nn.Module.__call__`, which runs `forward` plus a few extras (hooks, again). That's why you always write `model(x)`, never `model.forward(x)`.

In [6]:
# 🤔 Predict: nn.Module without super().__init__(). Which line fails?
class BrokenNN(nn.Module):
    def __init__(self):
        self.fc1 = nn.Linear(784, 50)


BrokenNN()

AttributeError: cannot assign module before Module.__init__() call

Read the bottom line: `cannot assign module before Module.__init__() call`. `nn.Module.__init__` creates the bookkeeping that records every layer you assign to `self`. Without it, `nn.Module` has nowhere to put `fc1`, so it refuses straight away. It's the same bug as `BrokenScale` in 01, just with a clearer message.

## 2. Data: a `Dataset` and a `DataLoader`

In [7]:
# ✍️ Images → float tensors in [0, 1], flattened to 784 numbers. 5,000 to train on, 1,000 to validate.
X = torch.tensor(data["X_train"], dtype=torch.float32).reshape(-1, 784) / 255
y = torch.tensor(data["y_train"], dtype=torch.long)
train_ds = TensorDataset(X[:5000], y[:5000])
val_ds = TensorDataset(X[5000:], y[5000:])
len(train_ds), len(val_ds)

(5000, 1000)

In [8]:
# 🤔 Predict: the shapes of the first batch, xb and yb?
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)
xb, yb = next(iter(train_loader))
xb.shape, yb.shape

(torch.Size([64, 784]), torch.Size([64]))

## 3. A plain PyTorch training loop

Run it, then read it. Each line is labelled:
- 🔬 **science**: the lines that make *this* model what it is
- 🔧 **plumbing**: the lines that would be the same for any model

In [9]:
torch.manual_seed(0)
model = NN(784, 10)                                          # 🔬
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)    # 🔬

for epoch in range(2):                                       # 🔧
    model.train()                                            # 🔧
    for xb, yb in train_loader:                              # 🔧
        scores = model(xb)                                   # 🔬
        loss = F.cross_entropy(scores, yb)                   # 🔬
        optimizer.zero_grad()                                # 🔧
        loss.backward()                                      # 🔧
        optimizer.step()                                     # 🔧

    model.eval()                                             # 🔧
    with torch.no_grad():                                    # 🔧
        val_losses = [F.cross_entropy(model(xv), yv).item() for xv, yv in val_loader]  # 🔬 + 🔧
    print(f"epoch {epoch}: last train loss {loss.item():.3f}, val loss {np.mean(val_losses):.3f}")  # 🔧

epoch 0: last train loss 0.508, val loss 0.656
epoch 1: last train loss 0.197, val loss 0.430


Count them: about 5 lines of science and 10 of plumbing. And this loop still doesn't use a GPU, save checkpoints, stop early, or log anything you could plot later. Every one of those features adds more plumbing.

### ✏️ Exercise 2: one more piece of plumbing (≈4 min)

Write `evaluate(model, loader)`. It returns the model's accuracy on `loader`, as a float between 0 and 1.

1. Put the model in evaluation mode with `model.eval()`.
2. Inside `with torch.no_grad():`, loop over the batches. For each batch, the predicted class is `model(xb).argmax(dim=1)`. Count how many predictions equal `yb` (`(preds == yb).sum().item()`), and how many there were (`len(yb)`).
3. Put the model back in training mode with `model.train()`, then return correct / total.

In [10]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += len(yb)
    model.train()
    return correct / total

In [11]:
# --- Check (do not edit) ---
val_acc = evaluate(model, val_loader)
assert val_acc is not None, "Finish the exercise above first ☝️ (evaluate returned None)"
assert 0.8 < val_acc <= 1.0, f"Expected an accuracy between 0.8 and 1 after 2 epochs, got {val_acc}"
assert model.training, "Put the model back in training mode with model.train() before returning"
print(f"✅ Validation accuracy: {val_acc:.3f}")

✅ Validation accuracy: 0.896


## Where we're going

Lightning's idea: **you write the 🔬 science as methods on a class, and Lightning owns the 🔧 plumbing.** It's the same deal as `TinyModule`: the framework calls methods that you override.

🔄 **Checkpoint:** run `git pull --no-edit` to get `tutoring/session01/solutions/02_bridge_to_pytorch.ipynb`.